In [ ]:
import pickle


load_file = "Daten/Lastreihe_CN_04-11_04_2022.pkl"
loaded_data = None
with open(load_file, "rb") as f:
        loaded_data = pickle.load(f)

In [ ]:
times = loaded_data["time"]
load_series = loaded_data["Lastreihe"]

In [ ]:
timestep_length = times[1] - times[0]
timeseries_length = len(times)

In [ ]:
# create daily heat profile, similar to a typical residential heat demand
daily_heat_profile = []
for t in range(96):
    if t < 24:
        daily_heat_profile.append(0.5 + 0.5 * (t / 24))
    elif t < 72:
        daily_heat_profile.append(1.0)
    else:
        daily_heat_profile.append(0.5 + 0.5 * ((96 - t) / 24))

# smooth the daily profile with a moving average
smoothed_profile = []
window_size = 5
for i in range(len(daily_heat_profile)):
    window = daily_heat_profile[max(0, i - window_size // 2):min(len(daily_heat_profile), i + window_size // 2 + 1)]
    smoothed_profile.append(sum(window) / len(window))

In [ ]:
# plot daily heat profile
import matplotlib.pyplot as plt
plt.plot(smoothed_profile)
plt.title("Daily Heat Profile")
plt.xlabel("Time (15-min intervals)")
plt.ylabel("Heat Demand (normalized)")
plt.grid()
plt.show()

In [ ]:
# insert the daily profile into the load series with some random variation
import random
final_heat_series = []
for t in range(timeseries_length):
    daily_index = t % 96
    variation = random.uniform(-0.1, 0.1)
    final_heat_series.append(smoothed_profile[daily_index] + variation)

In [ ]:
# plot first week of final heat series
plt.plot(final_heat_series[:96*7])
plt.title("Final Heat Demand Series (First Week)")
plt.xlabel("Time (15-min intervals)")
plt.ylabel("Heat Demand (normalized)")
plt.grid()
plt.show()

In [ ]:
# scale heat demand to realistic values (e.g., kW)
scaled_heat_series = [demand * 1000000 for demand in final_heat_series]

In [ ]:
# save final heat series to pickle file
output_file = "Daten/Heat_Demand_Series.pkl"
with open(output_file, "wb") as f:
    pickle.dump({
        "time": times,
        "heat_demand_series": scaled_heat_series
    }, f)